In [ ]:
%sql
USE CATALOG `ocb_datavault_${environment}_cleaned`;
USE SCHEMA `${target_schema}`;



CREATE OR REPLACE TABLE hub_callcenter (
    callcenter_hashkey string NOT NULL COMMENT 'Hashkey Hub Callcenter',
    business_key string COMMENT 'Định danh duy nhất cuộc gọi (timestamp + số queue – do South Telecom sinh)',
    source_event_date date COMMENT 'Trường kỹ thuật: Ngày sự kiện thay đổi có hiệu lực từ hệ thống nguồn.',
    load_timestamp timestamp COMMENT 'Trường kỹ thuật: Thời điểm bản ghi được nạp vào hệ thống nguồn.',
    record_source string COMMENT 'Trường kỹ thuật: Nguồn gốc của bản ghi dùng để xác định bảng nguồn dữ liệu.'
) CLUSTER BY AUTO;
ALTER TABLE hub_callcenter ALTER COLUMN callcenter_hashkey SET NOT NULL;
ALTER TABLE hub_callcenter ADD CONSTRAINT hub_callcenter_pk PRIMARY KEY (callcenter_hashkey);


CREATE OR REPLACE TABLE sat_callcenter_information (
    callcenter_hashkey string NOT NULL COMMENT 'Hashkey Hub Callcenter',
    hashdiff string COMMENT 'Hash diff của các cột satellite',
    source_event_date date COMMENT 'Trường kỹ thuật: Ngày sự kiện thay đổi có hiệu lực từ hệ thống nguồn.',
    load_timestamp timestamp COMMENT 'Trường kỹ thuật: Thời điểm bản ghi được nạp vào hệ thống nguồn.',
    record_source string COMMENT 'Trường kỹ thuật: Nguồn gốc của bản ghi dùng để xác định bảng nguồn dữ liệu.',
    uniqueid string COMMENT 'Mã duy nhất cuộc gọi – do hệ thống South Telecom sinh',
    gcalluuid string COMMENT 'Global call UUID: C0026@uniqueid – định danh toàn cầu trong hệ thống South Telecom',
    accountcode string COMMENT 'Đích đến cuối cùng: Extension/Queue/DID nhận cuộc gọi',
    dialid string COMMENT 'Mã DIALID từ CRM (CRM_yyyyMMddHHmmss|ProgramId|User) – dùng map với crm_contact_hist',
    sipserver string COMMENT 'Tên server SIP/node PBX xử lý cuộc gọi',
    src string COMMENT 'Số khởi tạo cuộc gọi – DID hoặc Extension (outbound)',
    dst string COMMENT 'Số đích cuộc gọi – số điện thoại khách hàng (outbound)',
    channel string COMMENT 'Kênh SIP phía nguồn cuộc gọi',
    dstchannel string COMMENT 'Kênh SIP phía đích cuộc gọi',
    queue string COMMENT 'Mã hàng đợi (format: C002610+số thứ tự, vd C00261003)',
    in_out string COMMENT 'Hướng cuộc gọi: out = outbound',
    did_number string COMMENT 'Đầu số DID của agent thực hiện gọi ra',
    prefix_detail string COMMENT 'prefix_detail',
    carrier string COMMENT 'Nhà mạng của cuộc gọi',
    callername string COMMENT 'Tên/mã agent thực hiện cuộc gọi',
    customer_id int COMMENT 'ID khách hàng trong database của South Telecom',
    customer_code string COMMENT 'Mã khách hàng (enriched từ DB South Telecom)',
    customername string COMMENT 'Tên khách hàng dựa vào số điện thoại (enriched)',
    customercif string COMMENT 'Số CIF khách hàng – trường multi-value, ETL cần tách chuỗi'
) CLUSTER BY AUTO;
ALTER TABLE sat_callcenter_information ALTER COLUMN callcenter_hashkey SET NOT NULL;
ALTER TABLE sat_callcenter_information ADD CONSTRAINT sat_callcenter_information_pk PRIMARY KEY (callcenter_hashkey);


CREATE OR REPLACE TABLE sat_callcenter_outcome (
    callcenter_hashkey string NOT NULL COMMENT 'Hashkey Hub Callcenter',
    hashdiff string COMMENT 'Hash diff của các cột satellite',
    source_event_date date COMMENT 'Trường kỹ thuật: Ngày sự kiện thay đổi có hiệu lực từ hệ thống nguồn.',
    load_timestamp timestamp COMMENT 'Trường kỹ thuật: Thời điểm bản ghi được nạp vào hệ thống nguồn.',
    record_source string COMMENT 'Trường kỹ thuật: Nguồn gốc của bản ghi dùng để xác định bảng nguồn dữ liệu.',
    calldate bigint COMMENT 'Ngày gọi (Calldate) – định dạng string',
    calldatetime string COMMENT 'Thời gian thực hiện cuộc gọi',
    createtime bigint COMMENT 'Thời điểm log cuộc gọi được ghi nhận (epoch)',
    duration int COMMENT 'Tổng thời lượng cuộc gọi (giây)',
    billsec int COMMENT 'Thời gian tính cước – nên dùng thay billtime (giây)',
    billtime int COMMENT 'Thời gian tính cước phí gọi',
    holdtime int COMMENT 'Thời gian giữ cuộc gọi (giây)',
    talktime int COMMENT 'Thời gian đàm thoại thực tế (giây)',
    waitingtime int COMMENT 'Thời gian chờ trước khi được tiếp nhận (giây)',
    moh_time int COMMENT 'Tổng thời gian Music on Hold (giây)',
    disposition string COMMENT 'Kết quả cuộc gọi (ANSWERED / NO ANSWER / BUSY / FAILED)',
    system_disposition string COMMENT 'Trạng thái hệ thống',
    extension_disposition string COMMENT 'Trạng thái của extension',
    calltype string COMMENT 'Loại cuộc gọi: makecall2 (click-to-call/thường), makecall3 (auto call), internal (nội bộ)',
    hangup_by string COMMENT 'Bên kết thúc cuộc gọi (accountcode nếu OCB, số KH nếu KH cúp máy)',
    connected string COMMENT 'Các line connect call',
    not_connected string COMMENT 'Các line not connect call',
    processed int COMMENT 'Quá trình',
    filename string COMMENT 'Đường dẫn file ghi âm cuộc gọi',
    moh_log string COMMENT 'Log chi tiết khi khách hàng nghe nhạc chờ',
    moh_log_process string COMMENT 'moh_log_process'
) CLUSTER BY AUTO;
ALTER TABLE sat_callcenter_outcome ALTER COLUMN callcenter_hashkey SET NOT NULL;
ALTER TABLE sat_callcenter_outcome ADD CONSTRAINT sat_callcenter_outcome_pk PRIMARY KEY (callcenter_hashkey);


CREATE OR REPLACE TABLE link_callcenter_customer (
    link_callcenter_customer_hashkey string NOT NULL COMMENT 'Link Hashkey',
    callcenter_hashkey string COMMENT 'FK đến hub_callcenter',
    customer_hashkey string COMMENT 'FK đến hub_customer (T24) – mapping qua customercif → CIF',
    source_event_date date COMMENT 'Trường kỹ thuật: Ngày sự kiện thay đổi có hiệu lực từ hệ thống nguồn.',
    record_source string COMMENT 'Trường kỹ thuật: Nguồn gốc của bản ghi dùng để xác định bảng nguồn dữ liệu.',
    load_timestamp timestamp COMMENT 'Trường kỹ thuật: Thời điểm bản ghi được nạp vào hệ thống nguồn.'
) CLUSTER BY AUTO;
ALTER TABLE link_callcenter_customer ALTER COLUMN link_callcenter_customer_hashkey SET NOT NULL;
ALTER TABLE link_callcenter_customer ADD CONSTRAINT link_callcenter_customer_pk PRIMARY KEY (link_callcenter_customer_hashkey);
